In [ ]:
import os

import pandas as pd

import paths as p

In [ ]:
logs_dir = p.DATA_DIR / "logs"
progress_file_name = "RZ_dj_progress.csv"  # downloaded from df pipeline by running progress_checker.py
recording_log_name = "recording_log.csv"  # downloaded from recording log from google drive

### load logs

dj progress log

In [ ]:
dj_progress = pd.read_csv(os.path.join(logs_dir, progress_file_name))

In [ ]:
dj_progress['date'] = pd.to_datetime(dj_progress['session_datetime']).dt.normalize()
dj_progress['mouse'] = dj_progress['subject']
dj_progress['insertion_number'] = dj_progress['insertion_number'].astype(int)

recording log

In [ ]:
recording_log = pd.read_csv(os.path.join(logs_dir, recording_log_name))

In [ ]:
recording_log = recording_log.drop(columns=['NIDAQ', 'simultaneous',
       'probe', 'depth', 'probe treatment', 'insertion speed',
       'resting time', 'surface', 'extraction speed', 'notes', 'rewards',
       'num trials', 'tw', 'current status', 'final data out'])
recording_log['date'] = pd.to_datetime(recording_log['date'], errors='coerce')
recording_log['insertion_number'] = recording_log['insertion_number'].astype(int)

### merging logs

In [ ]:
# Merge with left join on 3 key columns
cross_checked = recording_log.merge(
    dj_progress[['mouse', 'date', 'insertion_number', 'First_X_Column']],
    on=['mouse', 'date', 'insertion_number'],
    how='left'
)

# Label unmatched rows
cross_checked['First_X_Column'] = cross_checked['First_X_Column'].fillna('not_uploaded')

In [ ]:
cross_checked.to_csv(logs_dir / 'sessions_cross_checked.csv')

### examine cross check results

In [ ]:
sessions_cross_checked = pd.read_csv(os.path.join(logs_dir, 'sessions_cross_checked.csv'), index_col=0)

In [ ]:
sessions_cross_checked

In [ ]:
progress_overview = sessions_cross_checked['First_X_Column'].value_counts()
print(progress_overview)

In [ ]:
def get_stage_sessions(cross_checked, dj_df, stage):
    """Return dj_progress info for sessions stuck at a given pipeline stage."""
    rows = cross_checked[cross_checked['First_X_Column'] == stage]
    result = []
    for _, row in rows.iterrows():
        match = dj_df[
            (dj_df['mouse'] == row['mouse']) &
            (dj_df['date'] == pd.to_datetime(row['date'])) &
            (dj_df['insertion_number'] == row['insertion_number'])
        ]
        if not match.empty:
            info = match.iloc[0][['subject', 'session_datetime', 'insertion_number']].to_dict()
            result.append(info)
    return result

In [ ]:
for stage in ['SIClustering', 'SIExport', 'ManualCuration']:
    sessions = get_stage_sessions(sessions_cross_checked, dj_progress, stage)
    print(f"\n=== {stage} ({len(sessions)} sessions) ===")
    for s in sessions:
        print(s)

### Recording location summary per mouse

In [ ]:
sessions_cross_checked_exp2 = sessions_cross_checked.loc[
    ~sessions_cross_checked['sorting notes'].isin(['exp3', 'tester'])
].copy()

sessions_cross_checked_exp2['location'] = sessions_cross_checked_exp2['hemisphere'] + '_' + sessions_cross_checked_exp2['region']

location_order = ['l_str', 'r_str', 'l_v1', 'r_v1']

def sort_locations(locs):
    return sorted(locs, key=lambda x: location_order.index(x) if x in location_order else len(location_order))

location_summary = (
    sessions_cross_checked_exp2
    .groupby('mouse')['location']
    .apply(lambda x: sort_locations(x.unique()))
    .reset_index()
    .rename(columns={'location': 'unique_locations'})
)

session_counts = (
    sessions_cross_checked_exp2
    .groupby(['mouse', 'location'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[c for c in location_order if c in sessions_cross_checked_exp2['location'].unique()])
)

location_summary = location_summary.set_index('mouse').join(session_counts).reset_index()
location_summary.to_csv(logs_dir / 'location_summary.csv')

In [ ]:
location_summary